In [88]:
%matplotlib inline

# Preprocessing Data

Missing data

In [89]:
# Fetch credit-g dataset
from sklearn.datasets import fetch_openml
dataset_df = fetch_openml(name='autos', version=1, as_frame=True)

# show some random smaples
dataset_df.data.sample(5)

/opt/homebrew/lib/python3.11/site-packages/sklearn/datasets/_openml.py:1022: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,length,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
143,102.0,subaru,gas,std,four,sedan,fwd,front,97.2,172.0,...,108.0,mpfi,3.62,2.64,9.00,94.0,5200.0,26.0,32.0,9960.0
126,NaN,porsche,gas,std,two,hardtop,rwd,rear,89.5,168.9,...,194.0,mpfi,3.74,2.90,9.50,207.0,5900.0,17.0,25.0,32528.0
52,104.0,mazda,gas,std,two,hatchback,fwd,front,93.1,159.1,...,91.0,2bbl,3.03,3.15,9.00,68.0,5000.0,31.0,38.0,6795.0
21,118.0,dodge,gas,std,two,hatchback,fwd,front,93.7,157.3,...,90.0,2bbl,2.97,3.23,9.41,68.0,5500.0,37.0,41.0,5572.0
152,74.0,toyota,gas,std,four,hatchback,fwd,front,95.7,158.7,...,92.0,2bbl,3.05,3.03,9.00,62.0,4800.0,31.0,38.0,6488.0


In [90]:
# Check for missing data
print(dataset_df.data.isnull().sum())

normalized-losses    41
make                  0
fuel-type             0
aspiration            0
num-of-doors          2
body-style            0
drive-wheels          0
engine-location       0
wheel-base            0
length                0
width                 0
height                0
curb-weight           0
engine-type           0
num-of-cylinders      0
engine-size           0
fuel-system           0
bore                  4
stroke                4
compression-ratio     0
horsepower            2
peak-rpm              2
city-mpg              0
highway-mpg           0
price                 4
dtype: int64


In [91]:
# Remove values where less than 5% are missing
dataset_df.data.dropna(thresh=len(dataset_df.data)*0.95, axis=1, inplace=True)

/var/folders/xj/_rnfxgcx0j99crfb0488198w0000gn/T/ipykernel_15424/143097605.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_df.data.dropna(thresh=len(dataset_df.data)*0.95, axis=1, inplace=True)


In [92]:
# Convert categorical data to numerical data
import numpy as np

dataset_df.data["make"] = np.where(dataset_df.data["make"] == "Volkswagen", 1, 0)


/var/folders/xj/_rnfxgcx0j99crfb0488198w0000gn/T/ipykernel_15424/4025834287.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset_df.data["make"] = np.where(dataset_df.data["make"] == "Volkswagen", 1, 0)


In [93]:
# Imputing missing data
# Note: Split data before imputing!

# Often used strategies: mean, median and most_frequent

from sklearn.model_selection import train_test_split

# Split data (categories)
X_cat = dataset_df.data['num-of-doors'].values.reshape(-1, 1)
y = dataset_df.data['make'].values

print(X_cat.shape)
print(y.shape)

# Split data
X_cat_train, X_cat_test, y_train, y_test = train_test_split(X_cat, y, test_size=0.2, random_state=42)

print(X_cat_train)


(205, 1)
(205,)
[['four'], ['four'], ['four'], ['four'], ['two'], ..., ['two'], ['four'], ['four'], ['two'], ['four']]
Length: 164
Categories (2, object): ['four', 'two']


In [94]:
# Split data (numerical)
X_num = dataset_df.data['wheel-base'].values.reshape(-1, 1)

print(X_num.shape)
print(y.shape)

# Split data
X_num_train, X_num_test, y_train, y_test = train_test_split(X_num, y, test_size=0.2, random_state=42)


(205, 1)
(205,)


In [95]:
# Impute missing categorical data
from sklearn.impute import SimpleImputer

imp_cat = SimpleImputer(strategy='most_frequent')

X_cat_train = imp_cat.fit_transform(X_cat_train)
X_cat_test = imp_cat.transform(X_cat_test)

In [96]:
# Impute missing numerical data
import numpy as np

imp_num = SimpleImputer(strategy='mean') # is default

X_num_train = imp_num.fit_transform(X_num_train)
X_num_test = imp_num.transform(X_num_test)

X_train = np.append(X_cat_train, X_num_train, axis=1)
X_test = np.append(X_cat_test, X_num_test, axis=1)

In [97]:
# Replace 'make' with numerical values
from sklearn.preprocessing import OrdinalEncoder
dataset_df['make'] = OrdinalEncoder().fit_transform(dataset_df.data['make'].values.reshape(-1, 1))

In [98]:
#  Or via a pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import pandas as pd

df = pd.DataFrame(dataset_df.data)
X = df['wheel-base'].values.reshape(-1, 1)
y = df['make']

steps = [('imputation', SimpleImputer(strategy='most_frequent')),
         ('logical_regression', LogisticRegression())]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline(steps)
pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 0